<a href="https://colab.research.google.com/github/marcovirulucas/econ_apis/blob/main/world/IMF_01_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# International Monetary Fund API with Python
## Part 1 - Exploration
## CPI Dataset and Dimensions
------
*May 19, 2026*\
\
The International Monetary Fund API documentation can be found [here](https://data.imf.org/en/Resource-Pages/IMF-API), and a basic call structure is available [here](https://datasupport.imf.org/knowledge?id=knowledge_category&sys_kb_id=5ca208d697bc4390d4f0b437f053af9e&category_id=9959b2bc1b6391903dba646fbd4bcb6a).\
\
This script finds the datasets and parameters available in IMF Data Portal.

In [1]:
import sdmx
import pandas as pd

### Searching for Dataset

In [2]:
# Initialize the SDMX client configured to connect to the IMF Data server
IMF_DATA = sdmx.Client('IMF_DATA')

# Get all dataflows
f = IMF_DATA.dataflow()

# Search for datasets containing "CPI"
{i: v for i, v in f.dataflow.items() if 'CPI' in v.name['en']}

{'CPI_WCA_2026_FEB_VINTAGE': <DataflowDefinition IMF.STA:CPI_WCA_2026_FEB_VINTAGE(1.0.0): Consumer Price Index (CPI), World and Country Aggregates (CPI_WCA) 2026 February >,
 'CPI_2026_FEB_VINTAGE': <DataflowDefinition IMF.STA:CPI_2026_FEB_VINTAGE(2.0.0): Consumer Price Index (CPI) 2026 February>,
 'PIP': <DataflowDefinition IMF.STA:PIP(5.0.0): Portfolio Investment Positions by Counterpart Economy (formerly CPIS)>,
 'CPI': <DataflowDefinition IMF.STA:CPI(5.0.0): Consumer Price Index (CPI)>,
 'CPI_WCA': <DataflowDefinition IMF.STA:CPI_WCA(3.0.0): Consumer Price Index (CPI), World and Country Aggregates (CPI_WCA)>,
 'CPI_2026_JAN_VINTAGE': <DataflowDefinition IMF.STA:CPI_2026_JAN_VINTAGE(1.0.0): Consumer Price Index (CPI) 2026 January>,
 'CPI_2026_APR_VINTAGE': <DataflowDefinition IMF.STA:CPI_2026_APR_VINTAGE(1.0.0): Consumer Price Index (CPI) 2026 April>,
 'CPI_WCA_2026_APR_VINTAGE': <DataflowDefinition IMF.STA:CPI_WCA_2026_APR_VINTAGE(1.0.0): Consumer Price Index (CPI), World and Count

The search result shows the dataset ID (e.g., CPI) and a description.

In [3]:
# Obtain the metadata for the CPI dataset
f = IMF_DATA.dataflow('CPI')
print(f)

<sdmx.StructureMessage>
  <Header>
    id: 'IDREF7085'
    prepared: '2026-05-29T22:29:06.124341+00:00'
    sender: <Agency unknown>
    source: 
    test: False
  response: <Response [200]>
  Codelist (127): CL_ORGANIZATION CL_INSTR_ASSET CL_METHODOLOGY CL_DECI...
  ConceptScheme (7): IMF:CS_MASTER_DATA SDMX:SDMX_CONCEPT_ROLES IMF:CS_...
  DataflowDefinition (1): CPI
  DataStructureDefinition (1): DSD_CPI


After identifying a dataset, extract its structure to find the available dimensions. The code for this is shown under 'DataStructureDefinition'.

### Getting the dimensions of the dataset

In [4]:
# Extract the structure and access the dimensions that make up the structure
dsd = f.structure['DSD_CPI']
dsd.dimensions.components

[<Dimension COUNTRY>,
 <Dimension INDEX_TYPE>,
 <Dimension COICOP_1999>,
 <Dimension TYPE_OF_TRANSFORMATION>,
 <Dimension FREQUENCY>,
 <TimeDimension TIME_PERIOD>]

The results above show that CPI dataset has six dimensions: country, index type, expenditure category, transformation type, frequency and time period.

### Getting the codes for each dimension

Each dimension is called in a particular way (usually preceded by CL.). Search these names in the attributes of "codelist".

In [5]:
# Show all the attributes of "codelist", wich are the dimensions names
print(dir(f.codelist))

['CL_ACCESS_SHARING_LEVEL', 'CL_ACCOUNTING_ENTRY', 'CL_ACCOUNTS', 'CL_CIVIL_STATUS', 'CL_CLASSIFICATION_TYPE', 'CL_COFOG', 'CL_COICOP_1999', 'CL_COICOP_2018', 'CL_COMMODITY', 'CL_CONF_STATUS', 'CL_COUNTRY', 'CL_CPI_TYPE_OF_TRANSFORMATION', 'CL_CURRENCY', 'CL_DECIMALS', 'CL_DEPARTMENT', 'CL_DERIVATION_TYPE', 'CL_EXRATE', 'CL_FI_MATURITY', 'CL_FREQ', 'CL_FSENTRY', 'CL_FUNCTIONAL_CAT', 'CL_GENDER', 'CL_GFS_STO', 'CL_INDEX_TYPE', 'CL_INSTR_ASSET', 'CL_INT_ACC_ITEM', 'CL_INT_TTC', 'CL_LANGUAGE', 'CL_METHODOLOGY', 'CL_MFS_INSTR', 'CL_NA_STO', 'CL_OBS_STATUS', 'CL_ORGANIZATION', 'CL_OVERLAP', 'CL_PRICES', 'CL_REPORTING_PERIOD_TYPE', 'CL_SECTOR', 'CL_SEC_CLASSIFICATION', 'CL_SEX', 'CL_SOC_CONCEPTS', 'CL_STATISTICAL_MEASURES', 'CL_S_ADJUSTMENT', 'CL_TOPIC', 'CL_TRADE_FLOW', 'CL_TRANSFORMATION', 'CL_UNIT', 'CL_UNIT_MULT', 'CL_VALUATION', 'GF01', 'GF011', 'GF012', 'GF013', 'GF014', 'GF015', 'GF016', 'GF017', 'GF018', 'GF02', 'GF021', 'GF022', 'GF023', 'GF024', 'GF025', 'GF03', 'GF031', 'GF032', '

Here we look up available index types. The code "CPI" will be selected.

In [6]:
# Access the codes of the INDEX_TYPE dimension (and convert it into a Series)
codes = f.codelist.CL_INDEX_TYPE
res = sdmx.to_pandas(codes)
print(res)

CL_INDEX_TYPE
BSI                                                      Bank stock index
CCPI                                      Core consumer price index (CPI)
CCPISVN                            Core consumer price index (CCPI) (SVN)
CECO                    Commodity price index, energy, crude oil (petr...
CEMPI                                    Commodity net export price index
CEPI                                         Commodity export price index
CHICPIF                                      Core HICP inflation forecast
CMPI                                         Commodity import price index
CPCE                                                 Core PCE Price Index
CPI                                            Consumer price index (CPI)
CPIR                                  Relative consumer price index (CPI)
CPIU                                         Consumer price index (Urban)
CPIX                                                Commodity price index
D                       

We can also search within a codelist. In this case, we find the code for "CPI" in the index type list:

In [7]:
# Search an specific code
res_search = res.loc[res.str.contains('Consumer price index')]
print(res_search)

CL_INDEX_TYPE
CPI                              Consumer price index (CPI)
CPIU                           Consumer price index (Urban)
CPISRP    Consumer price index (CPI) common reference pe...
Name: Index type, dtype: object


Here we look up available expenditure categories. The code "_T" will be selected.

In [8]:
# Access the codes of the Expenditure Category dimension (and convert it into a Series)
codes = f.codelist.CL_COICOP_1999
res = sdmx.to_pandas(codes)
print(res)

CL_COICOP_1999
_T                                              All Items
CP01                     Food and non-alcoholic beverages
CP02           Alcoholic beverages, tobacco and narcotics
CP03                                Clothing and footwear
CP04     Housing, water, electricity, gas and other fuels
CP05    Furnishings, household equipment and routine h...
CP06                                               Health
CP07                                            Transport
CP08                                        Communication
CP09                               Recreation and culture
CP10                                            Education
CP11                               Restaurants and hotels
CP12                     Miscellaneous goods and services
CP13    Individual consumption expenditure of non-prof...
CP14    Individual consumption expenditure of general ...
Name: COICOP 1999, dtype: object


Here we look up available transformation types. The code "IX" will be selected.

In [9]:
# Access the codes of the TYPE_OF_TRANSFORMATION dimension (and convert it into a Series)
codes = f.codelist.CL_CPI_TYPE_OF_TRANSFORMATION
res = sdmx.to_pandas(codes)
print(res)

CL_CPI_TYPE_OF_TRANSFORMATION
IX                                                               Index
POP_PCH_PA_PT        Period average, Period-over-period percent change
YOY_PCH_PA_PT        Period average, Year-over-year (YOY) percent c...
WGT                                                             Weight
WGT_PT                                                 Weight, Percent
SRP_IX                     Standard reference period (2010=100), Index
SRP_POP_PCH_PA_PT    Standard reference period (2010=100), Period a...
SRP_YOY_PCH_PA_PT    Standard reference period (2010=100), Period a...
Name: Consumer Price Index (CPI) Type of Transformation, dtype: object


Here we look up available frequencies. The code "M" will be selected.

In [10]:
# Access the codes of the FREQUENCY dimension (and convert it into a Series)
codes = f.codelist.CL_FREQ
res = sdmx.to_pandas(codes)
print(res)

CL_FREQ
A                     Annual
D                      Daily
M                    Monthly
Q                  Quarterly
S      Half-yearly, semester
W                     Weekly
A2                  Biennial
A3                 Triennial
A4               Quadrennial
A5              Quinquennial
A10                Decennial
A20              Bidecennial
A30             Tridecennial
A_3       Three times a year
M2                 Bimonthly
M_2              Semimonthly
M_3      Three times a month
W2                  Biweekly
W3                 Triweekly
W4               Four-weekly
W_2               Semiweekly
W_3       Three times a week
D_2              Twice a day
H                     Hourly
H2                  Bihourly
H3                 Trihourly
B      Daily – business week
N                   Minutely
I                  Irregular
OA         Occasional annual
OM        Occasional monthly
_O                     Other
_U               Unspecified
_Z            Not applicable
Name: 

These codes will be necessary to construct the URL and make the data request.